# Contribution Margin Model - Dorje Teas

[SYNTHETIC DATA - Illustrative only. No internal Dorje data used.]

Purpose: reconcile order economics and identify products, channels, and order types that can support contribution-margin-positive D2C growth.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path('../04_data_model')
if not DATA_DIR.exists():
    DATA_DIR = Path('04_data_model')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'

def rupee(value):
    return f'Rs. {value:,.0f}'

orders = pd.read_csv(DATA_DIR / 'sample_orders.csv')
catalog = pd.read_csv(DATA_DIR / 'sample_product_catalog.csv')
orders['is_subscription_bool'] = orders['is_subscription'].astype(str).str.upper().eq('TRUE')
orders['is_gift_bool'] = orders['is_gift'].astype(str).str.upper().eq('TRUE')
orders['is_discounted'] = orders['discount_amount'] > 0
orders.head()

## Validate Margin Columns and Reconciliation

In [ ]:
calc_gross_margin = orders['net_revenue'] - orders['cogs'] - orders['packaging_cost'] - orders['shipping_cost'] - orders['gateway_fee']
orders['gross_margin_reconciliation_delta'] = orders['gross_margin'] - calc_gross_margin
orders['cm_reconciliation_delta'] = orders['contribution_margin'] - (orders['gross_margin'] - orders['attributed_ad_spend'])
assert orders['gross_margin_reconciliation_delta'].abs().max() <= 1
assert orders['cm_reconciliation_delta'].abs().max() <= 1
reconciliation = orders[['gross_margin_reconciliation_delta','cm_reconciliation_delta']].abs().describe()
reconciliation

## Aggregate Contribution Margin Waterfall

In [ ]:
waterfall = pd.DataFrame({
    'step': ['Gross Revenue','Discounts','COGS','Packaging','Shipping','Gateway Fee','Gross Margin','Attributed Ad Spend','Contribution Margin'],
    'amount': [
        orders['gross_revenue'].sum(),
        -orders['discount_amount'].sum(),
        -orders['cogs'].sum(),
        -orders['packaging_cost'].sum(),
        -orders['shipping_cost'].sum(),
        -orders['gateway_fee'].sum(),
        orders['gross_margin'].sum(),
        -orders['attributed_ad_spend'].sum(),
        orders['contribution_margin'].sum()
    ]
})
ax = sns.barplot(data=waterfall, x='step', y='amount', palette=['#4C78A8' if v >= 0 else '#E45756' for v in waterfall['amount']])
ax.set_title('Dorje Teas Contribution Margin Waterfall [SYNTHETIC DATA]')
ax.set_xlabel('Waterfall step')
ax.set_ylabel('Amount (Rs.)')
plt.xticks(rotation=45, ha='right')
for p in ax.patches:
    ax.annotate(f"{p.get_height()/1000:.0f}k", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom' if p.get_height() >= 0 else 'top', fontsize=8)
plt.tight_layout()
plt.show()
waterfall

## Contribution Margin Percentage by Product Category

In [ ]:
category_cm = orders.groupby('product_category', as_index=False).agg(net_revenue=('net_revenue','sum'), contribution_margin=('contribution_margin','sum'), orders=('order_id','count'))
category_cm['cm_pct'] = category_cm['contribution_margin'] / category_cm['net_revenue'] * 100
ax = sns.barplot(data=category_cm.sort_values('cm_pct', ascending=False), x='product_category', y='cm_pct', color='#54A24B')
ax.set_title('Dorje Teas CM% by Product Category [SYNTHETIC DATA]')
ax.set_xlabel('Product category')
ax.set_ylabel('Contribution margin (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
category_cm

## Contribution Margin by Channel

In [ ]:
channel_cm = orders.groupby('channel', as_index=False).agg(net_revenue=('net_revenue','sum'), contribution_margin=('contribution_margin','sum'), ad_spend=('attributed_ad_spend','sum'), orders=('order_id','count'))
channel_cm['cm_pct'] = channel_cm['contribution_margin'] / channel_cm['net_revenue'] * 100
ax = sns.barplot(data=channel_cm.sort_values('cm_pct', ascending=False), x='channel', y='cm_pct', color='#4C78A8')
ax.set_title('Dorje Teas CM% by Channel [SYNTHETIC DATA]')
ax.set_xlabel('Channel')
ax.set_ylabel('Contribution margin (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
channel_cm

## New vs Returning, Subscription, and Gift CM Comparisons

In [ ]:
def cm_summary(df, group_col):
    out = df.groupby(group_col, as_index=False).agg(net_revenue=('net_revenue','sum'), contribution_margin=('contribution_margin','sum'), orders=('order_id','count'))
    out['cm_pct'] = out['contribution_margin'] / out['net_revenue'] * 100
    return out

new_returning = cm_summary(orders, 'customer_type')
subscription = cm_summary(orders, 'is_subscription_bool')
gift = cm_summary(orders, 'is_gift_bool')
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.barplot(data=new_returning, x='customer_type', y='cm_pct', ax=axes[0], color='#4C78A8')
sns.barplot(data=subscription.assign(is_subscription_bool=subscription['is_subscription_bool'].map({True:'Subscription', False:'One-time'})), x='is_subscription_bool', y='cm_pct', ax=axes[1], color='#72B7B2')
sns.barplot(data=gift.assign(is_gift_bool=gift['is_gift_bool'].map({True:'Gift', False:'Non-gift'})), x='is_gift_bool', y='cm_pct', ax=axes[2], color='#F58518')
axes[0].set_title('New vs Returning CM%')
axes[1].set_title('Subscription vs One-Time CM%')
axes[2].set_title('Gift vs Non-Gift CM%')
for ax in axes:
    ax.set_ylabel('CM%')
fig.suptitle('Dorje Teas Contribution Margin Comparisons [SYNTHETIC DATA]')
plt.tight_layout()
plt.show()
new_returning, subscription, gift

## Discount Sensitivity Model

In [ ]:
discounts = np.arange(0, 0.21, 0.02)
sensitivity_rows = []
for _, row in catalog.iterrows():
    for discount in discounts:
        net_price = row['price'] * (1 - discount)
        gateway_fee = net_price * 0.02
        gross_margin = net_price - row['cogs'] - row['packaging_cost'] - row['shipping_cost_blended'] - gateway_fee
        sensitivity_rows.append({'product_name': row['product_name'], 'discount_pct': discount*100, 'gross_margin_pct': gross_margin / net_price * 100})
sensitivity = pd.DataFrame(sensitivity_rows)
top_products = ['First Flush Darjeeling','Original Darjeeling Chai','Selim Hill Gift Box Classic','Tea Club Darjeeling Black Monthly']
ax = sns.lineplot(data=sensitivity[sensitivity['product_name'].isin(top_products)], x='discount_pct', y='gross_margin_pct', hue='product_name', marker='o')
ax.set_title('Dorje Teas Discount Sensitivity by Product [SYNTHETIC DATA]')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Gross margin (%)')
plt.tight_layout()
plt.show()

## Break-Even CAC by Product

In [ ]:
catalog['break_even_cac'] = catalog['gross_margin_est']
breakeven = catalog[['product_name','category','price','gross_margin_est','gross_margin_pct','break_even_cac']].sort_values('break_even_cac', ascending=False)
ax = sns.barplot(data=breakeven, x='product_name', y='break_even_cac', color='#54A24B')
ax.set_title('Dorje Teas Break-Even CAC by Product [SYNTHETIC DATA]')
ax.set_xlabel('Product')
ax.set_ylabel('Break-even CAC (Rs.)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
breakeven

## Key Findings

- Contribution margin is the control point before scaling any paid acquisition.
- Returning and subscription-linked orders can carry stronger economics because they avoid first-order CAC pressure.
- Selim Hill Gift Box Classic and Selim Hill Gift Box Premium need separate packaging and shipping review because high AOV can hide margin stress.
- Discount sensitivity shows why premium products should not become discount-led acquisition products.

## Operator Decisions This Analysis Supports

- Scale products that can support paid acquisition on the first order.
- Use retention for products that need repeat purchase to justify CAC.
- Watch gift box margin after packaging and shipping, not just revenue.
- Do not scale campaigns with strong ROAS but weak contribution margin.

## What Real Dorje Data Would Improve

- Actual COGS, packaging, and shipping cost by SKU would replace the synthetic cost assumptions.
- Actual ad attribution would improve order-level paid media assignment.
- Actual refund and replacement costs would make the margin waterfall more complete.